#**Clasificación de opiniones sobre películas en imdb**

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing import sequence
import ipywidgets as widgets
from IPython.display import display
import numpy as np
import re
from tensorflow.keras.preprocessing import sequence

### Configuración y Entrenamiento del Modelo LSTM
En esta primera fase, importamos las librerías necesarias (TensorFlow, Keras, Widgets) y descargamos el dataset **IMDB**.
Entrenamos una red neuronal recurrente (**LSTM**) capaz de clasificar sentimientos (Positivo/Negativo) basándose en secuencias de texto.
* **max_features = 10000:** Limitamos el vocabulario a las 10,000 palabras más frecuentes.
* **maxlen = 100:** Normalizamos todas las reseñas a una longitud de 100 tokens.

In [ ]:
max_features = 10000  # Solo usamos las 10,000 palabras más frecuentes
maxlen = 100    # Cortamos las reseñas después de 100 palabras
batch_size = 32

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

In [ ]:
x_train

array([list([1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]),
       list([1, 194, 1153, 194, 8255, 78, 228,

In [ ]:
x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)
#Necesitamos rellenar las secuencias a una longitud fija, el maxlen

In [ ]:
model = Sequential([
    Embedding(max_features, 128, input_length=maxlen),
    LSTM(64, dropout=0.2),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.fit(x_train, y_train,
          batch_size=batch_size,
          epochs=3,
          validation_data=(x_test, y_test))

score, acc = model.evaluate(x_test, y_test, batch_size=batch_size)
print(f'Test accuracy: {acc:.4f}')

Epoch 1/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 85s 106ms/step - accuracy: 0.7458 - loss: 0.4958 - val_accuracy: 0.8453 - val_loss: 0.3651
Epoch 2/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 143s 107ms/step - accuracy: 0.8925 - loss: 0.2641 - val_accuracy: 0.8450 - val_loss: 0.3618
Epoch 3/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 81s 104ms/step - accuracy: 0.9289 - loss: 0.1887 - val_accuracy: 0.8455 - val_loss: 0.3973
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - accuracy: 0.8409 - loss: 0.4097
Test accuracy: 0.8455


#**El siguiente código nos permite recostruir una palabra a partir de su codificación numérica**

### Recuperación del Vocabulario
Para que nuestro script pueda "entender" lo que escribimos, necesitamos el diccionario original que asigna un número único a cada palabra inglesa.
Recuperamos el `word_index` de Keras para poder realizar la traducción inversa (Texto -> Números) manualmente.

In [ ]:
word_index = imdb.get_word_index()

In [ ]:
print(len(word_index))

88584


In [ ]:
word_index

{'fawn': 34701,
 'tsukino': 52006,
 'nunnery': 52007,
 'sonja': 16816,
 'vani': 63951,
 'woods': 1408,
 'spiders': 16115,
 'hanging': 2345,
 'woody': 2289,
 'trawling': 52008,
 "hold's": 52009,
 'comically': 11307,
 'localized': 40830,
 'disobeying': 30568,
 "'royale": 52010,
 "harpo's": 40831,
 'canet': 52011,
 'aileen': 19313,
 'acurately': 52012,
 "diplomat's": 52013,
 'rickman': 25242,
 'arranged': 6746,
 'rumbustious': 52014,
 'familiarness': 52015,
 "spider'": 52016,
 'hahahah': 68804,
 "wood'": 52017,
 'transvestism': 40833,
 "hangin'": 34702,
 'bringing': 2338,
 'seamier': 40834,
 'wooded': 34703,
 'bravora': 52018,
 'grueling': 16817,
 'wooden': 1636,
 'wednesday': 16818,
 "'prix": 52019,
 'altagracia': 34704,
 'circuitry': 52020,
 'crotch': 11585,
 'busybody': 57766,
 "tart'n'tangy": 52021,
 'burgade': 14129,
 'thrace': 52023,
 "tom's": 11038,
 'snuggles': 52025,
 'francesco': 29114,
 'complainers': 52027,
 'templarios': 52125,
 '272': 40835,
 '273': 52028,
 'zaniacs': 52130,

In [ ]:
# Invertimos el diccionario (Índice -> Palabra)
# Los índices de IMDB están desplazados por 3
reverse_word_index = {value + 3: key for (key, value) in word_index.items()}

# Rellenamos los índices especiales que usa Keras
reverse_word_index[0] = "<PAD>"    # Relleno (Padding)
reverse_word_index[1] = "<START>"  # Inicio de secuencia
reverse_word_index[2] = "<UNK>"    # Palabra desconocida (Unknown)
reverse_word_index[3] = "<UNUSED>" # No usada

In [ ]:
print("RECONSTRUCCIÓN DE LAS PRIMERAS 20 RESEÑAS \n")

for i in range(20):
    # Obtenemos la secuencia de números
    secuencia_numeros = x_train[i]

    # Convertimos números a palabras
    # Si hay un índice raro, pondrá '?'
    palabras_decodificadas = [reverse_word_index.get(num, '?') for num in secuencia_numeros]

    # Limpiamos para visualización
    # Quitamos los <PAD> para leerlo mejor.
    texto_legible = " ".join([word for word in palabras_decodificadas if word != "<PAD>"])

    # Obtenemos la etiqueta real (0 = Negativa, 1 = Positiva)
    etiqueta = "POSITIVA (1)" if y_train[i] == 1 else "NEGATIVA (0)"

    print(f"Reseña #{i+1} | Etiqueta Real: {etiqueta}")
    print(f"Texto recuperado: {texto_legible}")
    print("-" * 80)

RECONSTRUCCIÓN DE LAS PRIMERAS 20 RESEÑAS 

Reseña #1 | Etiqueta Real: POSITIVA (1)
Texto recuperado: cry at a film it must have been good and this definitely was also <UNK> to the two little boy's that played the <UNK> of norman and paul they were just brilliant children are often left out of the <UNK> list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't you think the whole story was so lovely because it was true and was someone's life after all that was shared with us all
--------------------------------------------------------------------------------
Reseña #2 | Etiqueta Real: NEGATIVA (0)
Texto recuperado: funny in equal <UNK> the hair is big lots of boobs <UNK> men wear those cut <UNK> shirts that show off their <UNK> sickening that men actually wore them and the music is just <UNK> trash that plays over and over again in almost every scene there is tr

In [ ]:
# CONFIGURACIÓN INICIAL
word_index = imdb.get_word_index()
max_features = 10000
maxlen = 100

### Motor de Preprocesamiento de Texto
Esta es la función crítica del ejercicio. Convierte una frase en inglés en un vector numérico válido para la red neuronal.
Pasos que realiza:
1.  **Limpieza:** Elimina caracteres raros y convierte a minúsculas.
2.  **Tokenización:** Separa la frase en palabras.
3.  **Indexación:** Busca cada palabra en el diccionario, le suma 3 (por los tokens reservados de Keras) y construye el vector.
    * Añade el token `[1]` al principio (<START>).
    * Si la palabra no existe o es poco frecuente, usa el token `2` (<UNK>).
4.  **Padding:** Rellena o recorta la secuencia para que tenga exactamente longitud 100.

In [ ]:
# FUNCIÓN DE PREPROCESAMIENTO
def procesar_texto_para_modelo(texto_bruto):
    # Limpieza básica: minúsculas y quitar caracteres no alfanuméricos
    texto_limpio = re.sub(r'[^\w\s]', '', texto_bruto.lower())
    palabras = texto_limpio.split()

    # Convertir palabras a índices
    # 0: Padding, 1: Start, 2: Unknown, 3: Unused
    vector = [1] # Iniciamos siempre con el token <START> (índice 1)

    for palabra in palabras:
        if palabra in word_index:
            indice = word_index[palabra] + 3
            # Solo aceptamos palabras dentro de las 10,000 más frecuentes
            if indice < max_features:
                vector.append(indice)
            else:
                vector.append(2) # <UNK> (Desconocido por ser poco frecuente)
        else:
            vector.append(2) # <UNK> (Palabra no existe en el diccionario)

    # Padding (Relleno) para ajustar a longitud 100
    vector_pad = sequence.pad_sequences([vector], maxlen=maxlen)
    return vector_pad

### Interfaz de Usuario (Data Entry)
Utilizamos `ipywidgets` para crear una pequeña interfaz gráfica dentro del cuaderno.
Esto nos permite probar el modelo interactivamente introduciendo nuestras propias reseñas en inglés sin necesidad de tocar código.

In [ ]:
# INTERFAZ GRÁFICA (WIDGETS)

texto_input = widgets.Text(
    value='',
    placeholder='Escribe tu reseña en inglés (ej: The movie was fantastic)',
    description='Reseña:',
    layout=widgets.Layout(width='80%') # Hacemos el campo más ancho
)

boton_prediccion = widgets.Button(
    description="Analizar Sentimiento",
    button_style='info', # Azul
    icon='search'
)

output_area = widgets.Output()

### Lógica de Inferencia y Prueba
Conectamos el botón a la función de predicción.
Al hacer clic:
1.  Captura el texto del widget.
2.  Lo convierte a vector usando nuestra función de preprocesamiento.
3.  Muestra el vector resultante (para verificar que la traducción numérica es correcta, como pide el enunciado).
4.  Pasa el vector por el modelo `model.predict()` e interpreta el resultado (Positivo/Negativo).

In [ ]:
def realizar_inferencia(b):
    with output_area:
        output_area.clear_output()
        entrada = texto_input.value

        if not entrada:
            print("Por favor escribe algo primero.")
            return

        # Convertimos texto a vector
        vector_procesado = procesar_texto_para_modelo(entrada)

        # Predicción del modelo
        prediccion = model.predict(vector_procesado, verbose=0)[0][0]

        # Mostramos resultado
        print(f"Texto original: '{entrada}'")
        print(f"Vector (primeros 10): {vector_procesado[0][:100]}...")
        print("-" * 30)

        # Interpretación (Sigmoide: 0 es Pésima, 1 es Excelente)
        if prediccion > 0.5:
            confianza = prediccion * 100
            print(f"OPINIÓN POSITIVA (Confianza: {confianza:.2f}%)")
        else:
            confianza = (1 - prediccion) * 100
            print(f"OPINIÓN NEGATIVA (Confianza: {confianza:.2f}%)")

boton_prediccion.on_click(realizar_inferencia)

print("Introduce una opinión sobre una película (en inglés) para clasificarla:")
display(texto_input, boton_prediccion, output_area)

# Frase negativa: "I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level.
# Frase negativa dataset: i was still suffering worst plot worst script worst movie i have ever seen i wanted to hit my head up against a wall for an hour
# Frase positiva: Zentropa is the most original movie I've seen in years. If you like unique thrillers that are influenced by film noir, then this is just the right cure for all of those Hollywood summer blockbusters clogging the theaters these days.
# Frase positiva dataset: that movie the people tell the story this does such a good job of capturing every moment of their involvement while we enter their world and feel every second

Introduce una opinión sobre una película (en inglés) para clasificarla:


Text(value='i was still suffering worst plot worst script worst movie i have ever seen i wanted to hit my head…

Button(button_style='info', description='Analizar Sentimiento', icon='search', style=ButtonStyle())

Output()

### Conclusión
Hemos logrado conectar un texto en lenguaje natural introducido por el usuario con un modelo de "Caja Negra" entrenado previamente. La clave ha sido reproducir fielmente el pipeline de preprocesamiento (limpieza, mapeo de índices y padding), lo que nos permite realizar inferencia en tiempo real sobre datos nuevos.